In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# ============================================================
# XGBOOST MANUAL TUNING - WEB ATTACK DETECTION 50K (GPU ACCELERATED)
# ============================================================

import json
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

RUN_ID = "XGB_RUN_003"

# -------- Paths --------
ARTIFACT_DIR = "./ML_Artifacts"
RESULT_DIR = "./XGBoost_Results"

os.makedirs(RESULT_DIR, exist_ok=True)

# ============================================================
# 2. HYPERPARAMETERS (รอบดัน Recall ของ Command Injection)
# ============================================================

N_ESTIMATORS = 150         # เพิ่มเป็น 150 ต้นเพื่อให้เก็บรายละเอียดรูปแบบ payload ได้มากขึ้น
MAX_DEPTH = 6              # ขยับความลึกเป็น 6 เพื่อจับเงื่อนไขคำสั่งผสมที่ซับซ้อน
LEARNING_RATE = 0.1        # ปรับ learning rate ให้อยู่ในระดับมาตรฐานเพื่อความแม่นยำ
SUBSAMPLE = 0.8
COLSAMPLE_BYTREE = 0.35    # เพิ่มการมองเห็นฟีเจอร์เป็น 35% (~10,500 มิติ) ช่วยดักจับคำสั่งเฉพาะ
MIN_CHILD_WEIGHT = 1
GAMMA = 0
REG_ALPHA = 0
REG_LAMBDA = 1
MAX_BIN = 32               # ล็อกไว้ที่ 32 เพื่อรักษาความเร็วบน VRAM 4GB ของ RTX 3050

# ============================================================
# 3. OTHER SETTINGS
# ============================================================

RANDOM_STATE = 42
N_JOBS = -1

CLASS_NAMES = [
    "SQLInjection",
    "XSS",
    "CommandInjection",
    "Normal"
]

# ============================================================
# 4. DISPLAY CONFIGURATION
# ============================================================

print("=" * 70)
print("XGBOOST EXPERIMENT CONFIGURATION")
print("=" * 70)

print(f"Run ID             : {RUN_ID}")
print(f"N Estimators       : {N_ESTIMATORS}")
print(f"Max Depth          : {MAX_DEPTH}")
print(f"Learning Rate      : {LEARNING_RATE}")
print(f"Subsample          : {SUBSAMPLE}")
print(f"Colsample Bytree   : {COLSAMPLE_BYTREE}")
print(f"Max Bin            : {MAX_BIN}")
print(f"Min Child Weight   : {MIN_CHILD_WEIGHT}")
print(f"Gamma              : {GAMMA}")
print(f"Reg Alpha          : {REG_ALPHA}")
print(f"Reg Lambda         : {REG_LAMBDA}")

print("=" * 70)

# ============================================================
# 5. LOAD ML ARTIFACTS
# ============================================================

print("\n" + "=" * 70)
print("LOADING ML ARTIFACTS")
print("=" * 70)

X_train_path = os.path.join(ARTIFACT_DIR, "X_train_tfidf.npz")
X_val_path = os.path.join(ARTIFACT_DIR, "X_val_tfidf.npz")
y_train_path = os.path.join(ARTIFACT_DIR, "y_train.npy")
y_val_path = os.path.join(ARTIFACT_DIR, "y_val.npy")

print("Loading X_train...")
X_train = sp.load_npz(X_train_path)

print("Loading X_val...")
X_val = sp.load_npz(X_val_path)

print("Loading y_train...")
y_train = np.load(y_train_path)

print("Loading y_val...")
y_val = np.load(y_val_path)

print("\nShapes:")
print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_val   : {y_val.shape}")

print("\n✓ Artifacts loaded successfully")

# ============================================================
# 6. CHECK DATA DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("DATA DISTRIBUTION")
print("=" * 70)

print("\nTraining Distribution:")
train_counts = pd.Series(y_train).value_counts().sort_index()
for label, count in train_counts.items():
    print(f"{CLASS_NAMES[int(label)]:20s}: {count:,}")

print("\nValidation Distribution:")
val_counts = pd.Series(y_val).value_counts().sort_index()
for label, count in val_counts.items():
    print(f"{CLASS_NAMES[int(label)]:20s}: {count:,}")

# ============================================================
# 7. CREATE XGBOOST MODEL (GPU ACCELERATED)
# ============================================================

print("\n" + "=" * 70)
print("CREATING XGBOOST MODEL")
print("=" * 70)

model = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
    subsample=SUBSAMPLE,
    colsample_bytree=COLSAMPLE_BYTREE,
    min_child_weight=MIN_CHILD_WEIGHT,
    gamma=GAMMA,
    reg_alpha=REG_ALPHA,
    reg_lambda=REG_LAMBDA,
    eval_metric="mlogloss",
    tree_method="hist",
    device="cuda:0",
    max_bin=MAX_BIN,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS
)

print("✓ Model created with GPU (CUDA) optimized settings")

# ============================================================
# 8. TRAINING
# ============================================================

print("\n" + "=" * 70)
print("TRAINING XGBOOST")
print("=" * 70)

start_time = time.time()

model.fit(
    X_train,
    y_train
)

training_time = time.time() - start_time

print("\n" + "=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)

print(f"Training Time : {training_time:.2f} seconds")
print(f"Training Time : {training_time / 60:.2f} minutes")

# ============================================================
# 9. VALIDATION PREDICTION
# ============================================================

print("\n" + "=" * 70)
print("VALIDATION EVALUATION")
print("=" * 70)

start_predict = time.time()
y_pred = model.predict(X_val)
prediction_time = time.time() - start_predict

print(f"Prediction Time : {prediction_time:.2f} seconds")

# ============================================================
# 10. METRICS
# ============================================================

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, average="macro", zero_division=0)
recall = recall_score(y_val, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_val, y_pred, average="macro", zero_division=0)

# ============================================================
# 11. DISPLAY MAIN RESULTS
# ============================================================

print("\n" + "=" * 70)
print("VALIDATION RESULTS")
print("=" * 70)

print(f"Accuracy        : {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Macro Precision : {precision:.4f} ({precision * 100:.2f}%)")
print(f"Macro Recall    : {recall:.4f} ({recall * 100:.2f}%)")
print(f"Macro F1        : {f1:.4f} ({f1 * 100:.2f}%)")

# ============================================================
# 12. CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

report = classification_report(
    y_val,
    y_pred,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
)
print(report)

# ============================================================
# 13. CONFUSION MATRIX
# ============================================================

print("=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

cm = confusion_matrix(y_val, y_pred)
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
print(cm_df)

# ============================================================
# 14. COMMAND INJECTION METRICS
# ============================================================

cmd_index = CLASS_NAMES.index("CommandInjection")
cmd_recall = recall_score(y_val, y_pred, labels=[cmd_index], average="macro", zero_division=0)
cmd_f1 = f1_score(y_val, y_pred, labels=[cmd_index], average="macro", zero_division=0)

print("\n" + "=" * 70)
print("COMMAND INJECTION PERFORMANCE")
print("=" * 70)

print(f"Recall : {cmd_recall:.4f} ({cmd_recall * 100:.2f}%)")
print(f"F1     : {cmd_f1:.4f} ({cmd_f1 * 100:.2f}%)")

# ============================================================
# 15. SAVE RESULTS
# ============================================================

print("\n" + "=" * 70)
print("SAVING RESULTS")
print("=" * 70)

result = {
    "run_id": RUN_ID,
    "model": "XGBoost",
    "dataset": "WebAttack_Cleaned_50K_v2",
    "split": {
        "train": int(len(y_train)),
        "validation": int(len(y_val))
    },
    "hyperparameters": {
        "n_estimators": N_ESTIMATORS,
        "max_depth": MAX_DEPTH,
        "learning_rate": LEARNING_RATE,
        "subsample": SUBSAMPLE,
        "colsample_bytree": COLSAMPLE_BYTREE,
        "min_child_weight": MIN_CHILD_WEIGHT,
        "gamma": GAMMA,
        "reg_alpha": REG_ALPHA,
        "reg_lambda": REG_LAMBDA,
        "max_bin": MAX_BIN
    },
    "metrics": {
        "accuracy": float(accuracy),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(f1),
        "command_injection_recall": float(cmd_recall),
        "command_injection_f1": float(cmd_f1)
    },
    "training_time_seconds": float(training_time),
    "prediction_time_seconds": float(prediction_time)
}

result_path = os.path.join(RESULT_DIR, f"{RUN_ID}_results.json")
with open(result_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=4, ensure_ascii=False)

# ============================================================
# 16. SAVE CONFUSION MATRIX
# ============================================================

cm_path = os.path.join(RESULT_DIR, f"{RUN_ID}_confusion_matrix.csv")
cm_df.to_csv(cm_path, encoding="utf-8-sig")

# ============================================================
# 17. SAVE MODEL
# ============================================================

model_path = os.path.join(RESULT_DIR, f"{RUN_ID}_model.json")
model.save_model(model_path)

# ============================================================
# 18. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)

print(f"Run ID          : {RUN_ID}")
print(f"Accuracy        : {accuracy * 100:.2f}%")
print(f"Macro F1        : {f1 * 100:.2f}%")
print(f"CMDI Recall     : {cmd_recall * 100:.2f}%")
print(f"Training Time   : {training_time:.2f} seconds ({training_time / 60:.2f} minutes)")

print(f"\nResults saved to:")
print(result_path)
print(cm_path)
print(model_path)

print("\n✓ XGBOOST EXPERIMENT COMPLETED")

XGBOOST EXPERIMENT CONFIGURATION
Run ID             : XGB_RUN_003
N Estimators       : 150
Max Depth          : 6
Learning Rate      : 0.1
Subsample          : 0.8
Colsample Bytree   : 0.35
Max Bin            : 32
Min Child Weight   : 1
Gamma              : 0
Reg Alpha          : 0
Reg Lambda         : 1

LOADING ML ARTIFACTS
Loading X_train...
Loading X_val...
Loading y_train...
Loading y_val...

Shapes:
X_train : (35000, 30000)
X_val   : (7500, 30000)
y_train : (35000,)
y_val   : (7500,)

✓ Artifacts loaded successfully

DATA DISTRIBUTION

Training Distribution:
SQLInjection        : 8,750
XSS                 : 8,750
CommandInjection    : 8,750
Normal              : 8,750

Validation Distribution:
SQLInjection        : 1,875
XSS                 : 1,875
CommandInjection    : 1,875
Normal              : 1,875

CREATING XGBOOST MODEL
✓ Model created with GPU (CUDA) optimized settings

TRAINING XGBOOST

TRAINING COMPLETED
Training Time : 32.75 seconds
Training Time : 0.55 minutes

VALIDA

In [2]:
# ============================================================
# CELL 2: XGBOOST FINAL TEST SET EVALUATION (STANDALONE)
# ============================================================

import os
import json
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# 1. CONFIGURATION
RUN_ID = "XGB_RUN_003"
ARTIFACT_DIR = "./ML_Artifacts"
RESULT_DIR = "./XGBoost_Results"
CLASS_NAMES = ["SQLInjection", "XSS", "CommandInjection", "Normal"]

print("=" * 70)
print("EVALUATING XGBOOST ON TEST SET")
print("=" * 70)

# 2. ตรวจสอบและดึงโมเดลมาใช้ (ถ้าไม่มีใน Memory ให้โหลดจากไฟล์ที่บันทึกไว้)
if 'model' not in locals():
    model_path = os.path.join(RESULT_DIR, f"{RUN_ID}_model.json")
    print(f"Loading trained model from: {model_path}")
    model = XGBClassifier()
    model.load_model(model_path)
    print("✓ Model loaded successfully")
else:
    print("✓ Using model currently in memory")

# 3. โหลดข้อมูล Test Set
X_test_path = os.path.join(ARTIFACT_DIR, "X_test_tfidf.npz")
y_test_path = os.path.join(ARTIFACT_DIR, "y_test.npy")

print("Loading Test Artifacts...")
X_test = sp.load_npz(X_test_path)
y_test = np.load(y_test_path)
print(f"X_test shape : {X_test.shape} | y_test shape : {y_test.shape}")

# 4. ทำนายผลบน Test Set
start_predict_test = time.time()
y_pred_test = model.predict(X_test)
prediction_time_test = time.time() - start_predict_test

# 5. คำนวณเมทริกซ์
acc_test = accuracy_score(y_test, y_pred_test)
prec_test = precision_score(y_test, y_pred_test, average="macro", zero_division=0)
rec_test = recall_score(y_test, y_pred_test, average="macro", zero_division=0)
f1_test = f1_score(y_test, y_pred_test, average="macro", zero_division=0)
f1_weighted_test = f1_score(y_test, y_pred_test, average="weighted", zero_division=0)

cmd_index = CLASS_NAMES.index("CommandInjection")
cmd_rec_test = recall_score(y_test, y_pred_test, labels=[cmd_index], average="macro", zero_division=0)
cmd_f1_test = f1_score(y_test, y_pred_test, labels=[cmd_index], average="macro", zero_division=0)

# 6. แสดงผลลัพธ์
print("\n" + "=" * 70)
print("TEST RESULTS")
print("=" * 70)
print(f"Accuracy         : {acc_test:.4f} ({acc_test * 100:.2f}%)")
print(f"Macro Precision  : {prec_test:.4f} ({prec_test * 100:.2f}%)")
print(f"Macro Recall     : {rec_test:.4f} ({rec_test * 100:.2f}%)")
print(f"Macro F1         : {f1_test:.4f} ({f1_test * 100:.2f}%)")
print(f"Weighted F1      : {f1_weighted_test:.4f} ({f1_weighted_test * 100:.2f}%)")
print(f"CMDI Recall      : {cmd_rec_test:.4f} ({cmd_rec_test * 100:.2f}%)")
print(f"CMDI F1          : {cmd_f1_test:.4f} ({cmd_f1_test * 100:.2f}%)")
print(f"Prediction Time  : {prediction_time_test:.2f} seconds")

print("\nClassification Report (Test):")
print(classification_report(y_test, y_pred_test, target_names=CLASS_NAMES, digits=4, zero_division=0))

cm_test = confusion_matrix(y_test, y_pred_test)
cm_test_df = pd.DataFrame(cm_test, index=CLASS_NAMES, columns=CLASS_NAMES)
print("Confusion Matrix (Test):")
print(cm_test_df)

# 7. บันทึกผลลัพธ์ Test Set
test_result = {
    "run_id": RUN_ID,
    "model": "XGBoost",
    "dataset": "Test",
    "metrics": {
        "accuracy": float(acc_test),
        "macro_precision": float(prec_test),
        "macro_recall": float(rec_test),
        "macro_f1": float(f1_test),
        "weighted_f1": float(f1_weighted_test),
        "command_injection_recall": float(cmd_rec_test),
        "command_injection_f1": float(cmd_f1_test)
    },
    "prediction_time_seconds": float(prediction_time_test)
}

test_result_path = os.path.join(RESULT_DIR, f"{RUN_ID}_test_results.json")
with open(test_result_path, "w", encoding="utf-8") as f:
    json.dump(test_result, f, indent=4, ensure_ascii=False)

test_cm_path = os.path.join(RESULT_DIR, f"{RUN_ID}_test_confusion_matrix.csv")
cm_test_df.to_csv(test_cm_path, encoding="utf-8-sig")

print(f"\n✓ Test Set results saved successfully to: {RESULT_DIR}")

EVALUATING XGBOOST ON TEST SET
Loading trained model from: ./XGBoost_Results\XGB_RUN_003_model.json
✓ Model loaded successfully
Loading Test Artifacts...
X_test shape : (7500, 30000) | y_test shape : (7500,)

TEST RESULTS
Accuracy         : 0.9824 (98.24%)
Macro Precision  : 0.9832 (98.32%)
Macro Recall     : 0.9824 (98.24%)
Macro F1         : 0.9825 (98.25%)
Weighted F1      : 0.9825 (98.25%)
CMDI Recall      : 0.9589 (95.89%)
CMDI F1          : 0.9774 (97.74%)
Prediction Time  : 0.28 seconds

Classification Report (Test):
                  precision    recall  f1-score   support

    SQLInjection     0.9951    0.9845    0.9898      1875
             XSS     0.9989    0.9909    0.9949      1875
CommandInjection     0.9967    0.9589    0.9774      1875
          Normal     0.9419    0.9952    0.9678      1875

        accuracy                         0.9824      7500
       macro avg     0.9832    0.9824    0.9825      7500
    weighted avg     0.9832    0.9824    0.9825      7500

Con